> **Status of this file:** code and methodology are complete; the notebook has not been executed yet. Run it from top to bottom (about 30 minutes, almost all of it in the two hyperparameter searches of Section 5). The commentary of each result is added after that run.

# TP1 · Task 4: Optimise the selected models, choose the decision threshold and evaluate once on the test period
## Classification of abnormal electric-vehicle charging sessions

**Course:** Mineração de Dados (MINDD), 2026/2027, 1st semester, ISEP / DEI

**Dataset:** educational version of the *China EV Charging Dataset* (Zhang et al., 2025). It has 441,077 charging sessions,
92 charging posts, 13 stations and 3 districts, from 2020 to 2021.

### Learning outcomes

- explain why hyperparameters are optimised and which ones are worth tuning (the *Parameter Optimisation* topic of the Ensemble Learning lecture);
- define an explicit search space with `scipy` distributions and run `RandomizedSearchCV` with a **temporal** cross-validation instead of a random one;
- keep the data that selects a configuration apart from the data that confirms it, and judge whether the optimised model improves **meaningfully and stably** on its initial configuration;
- choose a decision threshold from **out-of-fold** predictions with an operational criterion, freeze it, and apply it unchanged to the test period;
- analyse what changes between the default threshold (0.50) and the selected one: precision, recall, F1, alert rate and correct, false and missed alerts per 1,000 sessions;
- check whether the probabilities need calibration;
- interpret the final model with methods that suit it (grouped permutation importance and partial dependence) and state their limits.

### How to read this notebook

Task 4 is the first task that uses the **final test period** (from 2021-10-01). To keep that use honest, the notebook is organised in two blocks.
Sections 2 to 7 use **development data only** (2020-04-01 to 2021-09-30): they choose the models, search the hyperparameters, compare them with their
initial configurations and choose the threshold. At the end of Section 7 everything that was chosen is **frozen** and saved with a checksum.
Only then does Section 8 load the test sessions, once, and apply the frozen model and threshold. Section 9 interprets the final model.

| # | Section | Requirement of the Task 4 brief |
|---|---------|---------------------------------|
| 1 | Setup | — |
| 2 | Development data, folds and initial configurations | the "initial configurations" that the optimisation is compared with |
| 3 | Which models to optimise, and which parameters | a limited number (two or three) of promising models |
| 4 | Design of the search | training and validation data only, temporal order, explicit search space, a metric for the positive class |
| 5 | The searches | the selected configuration and the search procedure |
| 6 | Initial against optimised configuration | a meaningful and stable improvement? |
| 7 | Decision threshold and calibration | one criterion, chosen on out-of-fold predictions, then frozen |
| 8 | Final test evaluation | default against selected threshold, applied unchanged to the test |
| 9 | Interpretation of the final model | which variables and feature groups influence its predictions |
| 10 | Synthesis, saved files and decision log | — |

---
## 1. Setup

In [ ]:
import hashlib
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from IPython.display import display
from scipy.stats import loguniform, randint, spearmanr, uniform

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, brier_score_loss, f1_score, log_loss, precision_recall_curve,
                             roc_auc_score)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
DATA_PATH = Path("Charging_Data_educational.csv")
OUT_DIR = Path("prepared")
FIG_DIR = OUT_DIR / "figures_task4"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TARGET = "is_Abnormal"
NORMAL_CAUSES = {"Charging ends normally", "User stops charging"}      # every other end cause is abnormal (Task 1, Section 5)

# Everything that defines the evaluation comes from the files of Tasks 1 to 3, so the tasks cannot drift apart.
PROTOCOL = json.loads((OUT_DIR / "task2_protocol.json").read_text())
FEATURE_GROUPS = json.loads((OUT_DIR / "feature_groups.json").read_text())
TASK3 = json.loads((OUT_DIR / "task3_results.json").read_text())
BURN_IN_END = pd.Timestamp(PROTOCOL["temporal"]["burn_in_end"])
TEST_START = pd.Timestamp(PROTOCOL["temporal"]["test_start"])
ALERT_BUDGETS = tuple(PROTOCOL["metrics"]["operational"]["alert_budgets"])
GROUPS = FEATURE_GROUPS["groups"]
SELECTED = TASK3["selected_features"]                                   # the 27 features chosen by the ablation of Task 3
TOLERANCE = TASK3["tolerance"]                                          # a PR-AUC difference below this is treated as a tie (Task 3, Section 3.3)

# Search budget: number of random configurations per model. It is limited by the cost of one fit (Section 4).
N_ITER = {"Gradient boosting": 40, "Random forest": 20}
N_BOOTSTRAP = 200                                                       # resamples of the paired bootstrap of Section 6

sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold", "axes.titlesize": 11})
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 160)
C_NORMAL, C_ABN, C_GREY, C_ORANGE, C_GREEN, C_PURPLE = "#4C72B0", "#C44E52", "#7F7F7F", "#E1A730", "#2a9d8f", "#8172B3"

DECISIONS = []


def json_default(value):
    """JSON encoder for numpy scalars (numbers and booleans keep their type)."""
    return value.item() if hasattr(value, "item") else str(value)


def log_decision(section, topic, decision, evidence):
    """Store a decision together with its evidence (the full log is shown in Section 10)."""
    DECISIONS.append({"section": section, "topic": topic, "decision": decision, "evidence": evidence})


def savefig(fig, name):
    """Save a figure for the report and show it."""
    fig.savefig(FIG_DIR / f"{name}.png", bbox_inches="tight", dpi=150)
    plt.show()

---
## 2. Development data, folds and initial configurations

The function below reads the file and keeps only the sessions of one period. Called with `"development"`, it keeps the sessions that start between
2020-04-01 and 2021-09-30, exactly as in Tasks 2 and 3, and the test sessions are discarded at once, before any label or feature is created.
The same function will be called with `"test"` in Section 8, after everything has been frozen.

In [ ]:
RENAME = {   # same short names as in Tasks 1 to 3
    "UserID": "user_id", "Charging Post ID": "post_id", "Location Information": "location", "District Name": "district",
    "Start Time": "start_time", "End Time": "end_time", "Temperature(℃)": "temperature",
    "Relative Humidity(%)": "humidity", "Precipitation(mm)": "precipitation",
}
INVERSE_RENAME = {short: original for original, short in RENAME.items()}
DERIVED = {"hour": "Start Time", "dow": "Start Time", "rain_flag": "Precipitation(mm)"}      # features built from one column (Task 1, Section 12)
ALL_FEATURES = [f for group in GROUPS.values() for f in group]
NEEDED = {DERIVED.get(f, INVERSE_RENAME.get(f, f)) for f in ALL_FEATURES} | {"Start Time", "End Time", "end_cause"}


def load_sessions(period):
    """Sessions of one period with their features and label. 'development': start in [burn-in end, test start); 'test': start >= test start."""
    d = pd.read_csv(DATA_PATH, usecols=sorted(NEEDED)).rename(columns=RENAME)
    d["start_time"] = pd.to_datetime(d["start_time"], format="%Y-%m-%d %H:%M:%S")
    if period == "development":
        keep = (d["start_time"] >= BURN_IN_END) & (d["start_time"] < TEST_START)
    elif period == "test":
        keep = d["start_time"] >= TEST_START
    else:
        raise ValueError(period)
    d = d[keep].reset_index(drop=True)                                   # sessions of the other periods leave memory here
    for col in ("location", "district", "tariff_period", "end_cause"):
        d[col] = d[col].str.strip()
    d["end_time"] = pd.to_datetime(d["end_time"], format="%Y-%m-%d %H:%M:%S")
    d["hour"] = d["start_time"].dt.hour.astype("int8")
    d["dow"] = d["start_time"].dt.dayofweek.astype("int8")
    d["rain_flag"] = (d["precipitation"] > 0).astype("int8")
    d[TARGET] = (~d["end_cause"].isin(NORMAL_CAUSES)).astype("int8")
    return d


df = load_sessions("development")
print(f"Development data: {len(df):,} sessions ({df['start_time'].min().date()} to {df['start_time'].max().date()}), abnormal share {df[TARGET].mean():.2%}")
assert df["start_time"].max() < TEST_START and set(SELECTED) <= set(df.columns)

### 2.1 The four temporal folds

The folds are the expanding-window folds of Task 2: fold *k* trains on every labelled development session that started and ended before its
validation quarter, and validates on that quarter. The same integrity checks are repeated, because every result of this notebook depends on them.

In [ ]:
def temporal_folds(frame, protocol=PROTOCOL):
    """Expanding-window folds of Task 2: train = sessions that started AND ended before the validation quarter (purge),
    valid = sessions that started inside the quarter. Indices are positions in `frame`."""
    folds = []
    for spec in protocol["temporal"]["folds"]:
        lo, hi = pd.Timestamp(spec["validate_from"]), pd.Timestamp(spec["validate_before"])
        folds.append({"fold": spec["fold"], "lo": lo, "hi": hi,
                      "train": frame.index[(frame["start_time"] < lo) & (frame["end_time"] < lo)].to_numpy(),
                      "valid": frame.index[(frame["start_time"] >= lo) & (frame["start_time"] < hi)].to_numpy()})
    return folds


FOLDS = temporal_folds(df)
for f in FOLDS:
    tr, va = df.loc[f["train"]], df.loc[f["valid"]]
    assert tr["start_time"].min() >= BURN_IN_END and tr["end_time"].max() < f["lo"]
    assert va["start_time"].min() >= f["lo"] and va["start_time"].max() < f["hi"]
    assert np.intersect1d(f["train"], f["valid"]).size == 0
for a, b in zip(FOLDS[:-1], FOLDS[1:]):
    assert a["hi"] == b["lo"] and np.isin(a["train"], b["train"]).all()
for f, spec in zip(FOLDS, PROTOCOL["temporal"]["folds"]):
    assert len(f["train"]) == spec["train_sessions"] and len(f["valid"]) == spec["validation_sessions"]      # same folds as Task 2
assert FOLDS[-1]["hi"] == TEST_START
print("The four folds match the Task 2 protocol and pass the temporal integrity checks.")

fold_table = pd.DataFrame([{"fold": f["fold"], "validation quarter": f"{f['lo'].date()} to {(f['hi'] - pd.Timedelta(days=1)).date()}",
                            "train sessions": len(f["train"]), "validation sessions": len(f["valid"]),
                            "validation abnormal share": df.loc[f["valid"], TARGET].mean()} for f in FOLDS]).set_index("fold")
display(fold_table.style.format({"train sessions": "{:,.0f}", "validation sessions": "{:,.0f}", "validation abnormal share": "{:.1%}"}))

### 2.2 Models, metrics and the initial configurations

The two models that Task 3 ranked first are rebuilt here with the **initial configurations that Task 3 selected**: gradient boosting with depth 6, learning rate 0.1
and 200 iterations, and the random forest with 200 trees and at least 50 sessions per leaf (`TASK3["selected_configurations"]`). Both use the tree view of Task 1
(ordinal codes for station and district, missing values kept) and the 27 selected features. Boosting runs without early stopping, because the internal early
stopping splits the training rows at random and would let later sessions decide when to stop for earlier ones (Task 3, Section 5.1).

The metric functions are those of Tasks 2 and 3. The new function `threshold_report` computes, for any threshold, the measures that the Task 4 brief asks us to
compare: precision, recall, F1, alert rate and the correct, false and missed alerts per 1,000 sessions.

In [ ]:
INITIAL = {"Gradient boosting": dict(max_depth=6, learning_rate=0.1, max_iter=200, min_samples_leaf=100),
           "Random forest": dict(n_estimators=200, min_samples_leaf=50, max_features="sqrt")}
assert TASK3["selected_configurations"]["Gradient boosting"] == "HGB depth=6, lr=0.1, 200 it"
assert TASK3["selected_configurations"]["Random forest"] == "RF leaf=50"
CATEGORICAL_IN_USE = [c for c in SELECTED if c in FEATURE_GROUPS["categorical"]]


def tree_view(features):
    """Preprocessing of tree-based models (Task 1, Section 13): ordinal codes for the categories (unseen category -> -1, treated as missing),
    everything else untouched, and missing values kept as NaN."""
    cat = [c for c in features if c in FEATURE_GROUPS["categorical"]]
    rest = [c for c in features if c not in cat]
    steps = [("num", "passthrough", rest)]
    if cat:
        steps.insert(0, ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1, dtype=float), cat))
    return ColumnTransformer(steps, verbose_feature_names_out=False).set_output(transform="pandas")


def make_boosting(**params):
    """Gradient boosting pipeline. Unspecified parameters keep the initial configuration of Task 3."""
    p = {**INITIAL["Gradient boosting"], "early_stopping": False, "random_state": SEED, **params}
    return Pipeline([("prep", tree_view(SELECTED)),
                     ("clf", HistGradientBoostingClassifier(categorical_features=CATEGORICAL_IN_USE or None, **p))])


def make_forest(**params):
    """Random forest pipeline. Unspecified parameters keep the initial configuration of Task 3."""
    p = {**INITIAL["Random forest"], "n_jobs": -1, "random_state": SEED, **params}
    return Pipeline([("prep", tree_view(SELECTED)), ("clf", RandomForestClassifier(**p))])


BUILDERS = {"Gradient boosting": make_boosting, "Random forest": make_forest}


def ranking_metrics(y, p):
    """Metrics that do not depend on a threshold (Task 2, Section 6)."""
    y = np.asarray(y)
    return {"PR-AUC": average_precision_score(y, p), "lift": average_precision_score(y, p) / y.mean(), "ROC-AUC": roc_auc_score(y, p),
            "Brier": brier_score_loss(y, p), "log-loss": log_loss(y, np.clip(p, 1e-6, 1 - 1e-6))}


def threshold_report(y, p, threshold):
    """What a fixed threshold does: sessions with probability >= threshold raise an alert. All counts are per 1,000 sessions."""
    y, p = np.asarray(y), np.asarray(p)
    alert = p >= threshold
    tp, fp = int((alert & (y == 1)).sum()), int((alert & (y == 0)).sum())
    fn = int(((~alert) & (y == 1)).sum())
    n = len(y)
    precision = tp / (tp + fp) if tp + fp else float("nan")
    recall = tp / (tp + fn) if tp + fn else float("nan")
    f1 = 2 * precision * recall / (precision + recall) if tp else 0.0
    return {"threshold": threshold, "precision": precision, "recall": recall, "F1": f1, "alert rate": alert.mean(),
            "correct alerts per 1,000": 1000 * tp / n, "false alerts per 1,000": 1000 * fp / n, "missed per 1,000": 1000 * fn / n}


def alert_metrics(y, score, budget, rng):
    """Flag the top `budget` share of sessions by score (ties broken at random): the operational measures of Task 2, Section 6."""
    n = len(y)
    k = int(round(budget * n))
    order = np.lexsort((rng.random(n), -score))
    flagged = np.zeros(n, dtype=bool)
    flagged[order[:k]] = True
    tp = int((y[flagged] == 1).sum())
    return {"precision": tp / k, "recall": tp / y.sum(), "correct alerts per 1,000": 1000 * tp / n,
            "false alerts per 1,000": 1000 * (k - tp) / n, "missed per 1,000": 1000 * (int(y.sum()) - tp) / n}


def summarise_folds(values, weights):
    """Aggregate one metric over the temporal folds (Task 2, Section 8)."""
    v = pd.Series(values, dtype=float)
    return pd.Series({"mean (unweighted)": v.mean(), "mean (weighted by validation sessions)": np.average(v, weights=weights),
                      "std across folds": v.std(ddof=1), "worst fold": v.min(), "best fold": v.max(),
                      "trend per fold": np.polyfit(np.arange(1, len(v) + 1), v.to_numpy(), 1)[0]})


OOF = {}          # out-of-fold predictions: OOF[label][fold] = probabilities of the validation quarter of that fold
FIT_SECONDS = {}  # mean seconds to fit one fold


def run_folds(label, make_model, folds=FOLDS):
    """Fit a fresh pipeline inside each training fold and keep the predictions of its validation quarter (Task 2 protocol)."""
    OOF[label], seconds = {}, []
    for f in folds:
        tr, va = df.loc[f["train"]], df.loc[f["valid"]]
        started = time.time()
        model = make_model().fit(tr[SELECTED], tr[TARGET])
        OOF[label][f["fold"]] = model.predict_proba(va[SELECTED])[:, 1]
        seconds.append(time.time() - started)
    FIT_SECONDS[label] = float(np.mean(seconds))
    return pd.Series({f["fold"]: average_precision_score(df.loc[f["valid"], TARGET], OOF[label][f["fold"]]) for f in folds}, name=label)


def pr_auc_by_fold(label):
    return pd.Series({f["fold"]: average_precision_score(df.loc[f["valid"], TARGET], OOF[label][f["fold"]]) for f in FOLDS}, name=label)


t0 = time.time()
initial_scores = pd.DataFrame({f"{name} (initial)": run_folds(f"{name} (initial)", BUILDERS[name]) for name in INITIAL})
print(f"initial configurations fitted on the four folds in {time.time() - t0:.0f} s")
display(initial_scores.T.rename(columns=lambda c: f"fold {c}").assign(mean=lambda t: t.mean(axis=1)).style.format("{:.3f}"))

task3_means = {m["model"].split(" (")[0]: m["PR-AUC mean"] for m in TASK3["comparison"]}
for name in INITIAL:
    reproduced = initial_scores[f"{name} (initial)"].mean()
    print(f"{name}: mean PR-AUC {reproduced:.4f} here, {task3_means[name]:.4f} in Task 3")
    assert abs(reproduced - task3_means[name]) < 0.002, "the initial configurations must reproduce Task 3"

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

---
## 3. Which models to optimise, and which parameters

### 3.1 Choosing a limited number of promising models

The brief asks for two or three models. Optimisation costs many fits, so it should go where the gain can be. Task 3 gives the evidence. A model is a candidate when
its mean PR-AUC is within the tolerance of the best one (a tie with the best cannot be told from it) and when it brings a different ensemble principle, so
that the optimisation does not repeat itself.

In [ ]:
comparison3 = pd.DataFrame(TASK3["comparison"])
comparison3["model"] = comparison3["model"].str.split(" \\(").str[0]
comparison3 = comparison3.set_index("model")[["PR-AUC mean", "Brier", "log-loss", "seconds per fold"]]
best3 = comparison3["PR-AUC mean"].max()
comparison3["gap to best"] = comparison3["PR-AUC mean"] - best3
reasons = {
    "Gradient boosting": "Best model of Task 3 (boosting: reduces bias). Candidate.",
    "Random forest": "Within the tolerance of the best (bagging: reduces variance). Candidate.",
    "Stacking": "0.015 below the best; its constituents (logistic regression, Naive Bayes, tree) are not tuned here, and tuning the combination would multiply the cost.",
    "Decision tree": "A single tree: it is a constituent of the forest and 0.03 below the best.",
    "Logistic regression": "The penalty has no visible effect (Task 3, Section 4.3), so there is nothing to gain from a search.",
    "Naive Bayes": "Probabilities worse than a constant prediction (Task 3, Section 4.3); no complexity setting to tune.",
    "B1 post failure rate": "Baseline, not a model.", "B0 no skill": "Baseline, not a model.",
}
comparison3["decision"] = ["optimise" if abs(g) <= TOLERANCE or m in INITIAL else "no" for m, g in comparison3["gap to best"].items()]
comparison3["reason"] = [reasons[m] for m in comparison3.index]
display(comparison3.drop(columns=["Brier", "log-loss"]).style.format({"PR-AUC mean": "{:.3f}", "seconds per fold": "{:.1f}", "gap to best": "{:+.3f}"}, na_rep="—")
        .set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
assert list(comparison3.index[comparison3["decision"] == "optimise"]) == ["Gradient boosting", "Random forest"]

**Decision.** We optimise **two** models, gradient boosting and the random forest. They are tied at the top of Task 3 (a difference of 0.003, below the tolerance of 0.005) and
they represent different ensemble principles. The other models are left out for the reasons in the last column: either they are far from the best, or Task 3 already
showed that their one complexity setting does not matter.

### 3.2 Understanding the parameters

The *Parameter Optimisation* lecture starts from the parameters of the algorithm and warns that "not all are necessary to tune". The cell shows how many parameters each model has.

In [ ]:
for name, model in [("HistGradientBoostingClassifier", make_boosting().named_steps["clf"]), ("RandomForestClassifier", make_forest().named_steps["clf"])]:
    params = model.get_params()
    print(f"{name}: {len(params)} parameters")

Following the lecture, we tune only the parameters that control the complexity of the model and the amount of randomness in it, and we leave the rest at
values that we can justify. The table lists them for both models.

| Parameter | Model | What it controls | Initial value | Searched? | Reason |
|-----------|-------|------------------|---------------|-----------|--------|
| `learning_rate` | boosting | step of each new tree: small values need more trees | 0.1 | yes | The lecture lists it first. It trades off with the number of iterations |
| `max_iter` | boosting | number of boosting rounds (the lecture's `n_estimators`) | 200 | yes | Too few underfit, too many overfit. Early stopping would normally choose it, but it is not available without a random split, so we search it |
| `max_depth`, `max_leaf_nodes` | boosting | size of each tree | 6 and 31 (default) | yes | They control the complexity of each tree |
| `min_samples_leaf` | boosting | smallest leaf | 100 | yes | The main protection against overfitting small groups of sessions |
| `l2_regularization` | boosting | penalty on the leaf values | 0 (default) | yes | The regularisation term of the lecture |
| `max_features` | boosting | share of the features seen at each split | 1.0 (default) | yes | The *feature fraction* of the lecture. It adds randomness and speed |
| `n_estimators` | forest | number of trees | 200 | yes, three values | The lecture searches it; more trees help up to a plateau and cost time |
| `max_depth` | forest | depth of each tree | none | yes | Lecture values: none, 10, 20, 30 |
| `min_samples_leaf` | forest | smallest leaf | 50 | yes | Task 3 found 50 better than 200, so the useful region is below 200 |
| `max_features` | forest | features tried at each split | √27 (about 5) | yes | It sets how different the trees are (the lecture's *feature randomness*) |
| `max_samples` | forest | share of sessions in each bootstrap sample | all | yes | It sets how different the trees are (the lecture's *sample randomness*) and shortens the fit |
| `early_stopping` | boosting | stops adding trees on a validation split | off | no | The internal split is random: it breaks temporal order |
| `class_weight` | both | weights of the two classes | none | no | Task 1 (Section 13) showed that weights distort the probabilities that the threshold and the calibration check need |
| `loss`, `criterion`, `bootstrap` and the rest | both | | defaults | no | Standard choices for a binary target, not relevant for the minority class |

---
## 4. Design of the search

The brief fixes four conditions for the search: it uses only training and validation data, it respects temporal order, it has an explicit search space and it
optimises a metric that suits the positive class. This section turns them into a procedure.

### 4.1 Which data select, which data confirm

The four expanding-window folds of Task 2 are the only validation data. Task 3 used all four to choose the initial configurations, and if we used all four again
to choose the optimised ones and then compared the two on the same folds, the comparison would favour the optimised configuration by construction: a search
over many configurations always finds some that look better on the data it saw.

We therefore split the roles:
* the **search folds** are folds 1 to 3 (validation quarters 2020-Q4, 2021-Q1 and 2021-Q2). The search scores each configuration on them and nothing else;
* the **confirmation fold** is fold 4 (validation quarter 2021-Q3). The search never sees it, so the comparison between the initial and the optimised configuration on this
  fold is an honest estimate of what the optimisation brings, and it is the quarter closest to the test period.

The scores are computed by `RandomizedSearchCV` with a **list of temporal splits** as `cv`: each split trains on the sessions before a quarter and validates on that quarter,
so training always precedes validation and sessions that were still running at the boundary are purged, as in Task 2. A random K-fold, as in the lecture's example, is not used,
because it would let future sessions train the model that predicts earlier ones.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
d2n = mdates.date2num
roles = {1: C_ORANGE, 2: C_ORANGE, 3: C_ORANGE, 4: C_PURPLE}
for i, f in enumerate(FOLDS):
    y0 = len(FOLDS) - i
    ax.broken_barh([(d2n(BURN_IN_END), d2n(f["lo"]) - d2n(BURN_IN_END))], (y0 - 0.3, 0.6), color=C_NORMAL, alpha=0.85)
    ax.broken_barh([(d2n(f["lo"]), d2n(f["hi"]) - d2n(f["lo"]))], (y0 - 0.3, 0.6), color=roles[f["fold"]])
ax.broken_barh([(d2n(TEST_START), d2n(pd.Timestamp("2022-01-01")) - d2n(TEST_START))], (-0.3, 0.6), color=C_GREEN, hatch="///", alpha=0.55)
ax.set_yticks(list(range(len(FOLDS), 0, -1)) + [0])
ax.set_yticklabels([f"fold {f['fold']}" for f in FOLDS] + ["final test"])
ax.xaxis_date(); ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y")); ax.grid(axis="y", visible=False)
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=C_NORMAL, alpha=0.85), plt.Rectangle((0, 0), 1, 1, color=C_ORANGE), plt.Rectangle((0, 0), 1, 1, color=C_PURPLE),
                   plt.Rectangle((0, 0), 1, 1, color=C_GREEN, alpha=0.55, hatch="///")],
          labels=["training (expanding)", "search folds: select the configuration", "confirmation fold: never seen by the search", "final test: untouched until Section 8"],
          loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, fontsize=8)
ax.set_title("Roles of the temporal folds in Task 4")
savefig(fig, "04_1_search_design")

### 4.2 Search method, metric, budget and selection rule

**Method.** The lecture presents three methods. *Grid search* tries every combination: for the boosting space below it would need an enormous number of fits, since
several parameters are continuous. *Bayesian optimisation* (`BayesSearchCV` of `scikit-optimize`) is efficient but needs an extra library that the project does not use.
*Random search* samples configurations from the distributions of an explicit space, "is often more efficient and can find good solutions faster" (lecture), and needs
only `scikit-learn` and `scipy`. We use `RandomizedSearchCV` with `scipy.stats` distributions, as in the lecture's example.

**Metric.** `scoring="average_precision"` is the PR-AUC, the primary metric of the Task 2 protocol. It is a metric of the positive class (it ignores true negatives), it does not
depend on a threshold, and `RandomizedSearchCV` averages it over the search folds without weights, which is the aggregation rule of Task 2 (Section 8).

**Budget.** 40 configurations for boosting and 20 for the forest, each fitted on 3 folds. The forest is more expensive (about 10 seconds per fit against about 2.5), so
it gets fewer configurations. With a fixed budget, random search covers each parameter more densely than a grid would.

**Selection rule.** We keep the configuration with the highest mean PR-AUC on the search folds. If several are within the tolerance of 0.005 of it (they cannot be told
apart), we keep the one with the **shortest fit time**, because the cost of fitting is what a simpler configuration saves. This is the rule of Tasks 2 and 3 (*the simpler and more
stable model*), made operational with a quantity that the search measures.

### 4.3 The search spaces

Both spaces are written out explicitly. The ranges extend the values of the lecture (and of the initial configuration) in both directions, so that the initial
configuration lies inside the space and the search can move away from it.

In [ ]:
SPACES = {
    "Gradient boosting": {
        "clf__learning_rate": loguniform(0.02, 0.3),            # log scale: the effect of a step is multiplicative
        "clf__max_iter": randint(100, 501),                      # 100 to 500 boosting rounds
        "clf__max_depth": [None, 3, 4, 6, 8, 10],                # None = limited only by max_leaf_nodes
        "clf__max_leaf_nodes": randint(8, 64),                   # 8 to 63 leaves per tree
        "clf__min_samples_leaf": randint(20, 401),               # 20 to 400 sessions per leaf
        "clf__l2_regularization": [0.0, 0.01, 0.1, 1.0, 10.0],   # 0 = no penalty (initial value)
        "clf__max_features": uniform(0.5, 0.5),                  # 50% to 100% of the features at each split
    },
    "Random forest": {
        "clf__n_estimators": [100, 200, 300],
        "clf__max_depth": [None, 10, 20, 30],                    # lecture values
        "clf__min_samples_leaf": randint(10, 201),               # 10 to 200 sessions per leaf
        "clf__max_features": ["sqrt", "log2", 0.5],              # about 5, 4 or 13 of the 27 features
        "clf__max_samples": uniform(0.4, 0.6),                   # 40% to 100% of the sessions per tree
    },
}
space_table = pd.DataFrame([
    ("Gradient boosting", "learning_rate", "log-uniform", "0.02 to 0.3"), ("Gradient boosting", "max_iter", "integer, uniform", "100 to 500"),
    ("Gradient boosting", "max_depth", "choice", "none, 3, 4, 6, 8, 10"), ("Gradient boosting", "max_leaf_nodes", "integer, uniform", "8 to 63"),
    ("Gradient boosting", "min_samples_leaf", "integer, uniform", "20 to 400"), ("Gradient boosting", "l2_regularization", "choice", "0, 0.01, 0.1, 1, 10"),
    ("Gradient boosting", "max_features", "uniform", "0.5 to 1.0"),
    ("Random forest", "n_estimators", "choice", "100, 200, 300"), ("Random forest", "max_depth", "choice", "none, 10, 20, 30"),
    ("Random forest", "min_samples_leaf", "integer, uniform", "10 to 200"), ("Random forest", "max_features", "choice", "sqrt, log2, 0.5"),
    ("Random forest", "max_samples", "uniform", "0.4 to 1.0")], columns=["model", "parameter", "distribution", "range"])
display(space_table.style.hide(axis="index"))

SEARCH_FOLDS, CONFIRMATION_FOLD = FOLDS[:3], FOLDS[3]
TEMPORAL_CV = [(f["train"], f["valid"]) for f in SEARCH_FOLDS]           # (train positions, validation positions) for each search fold
assert all(len(set(tr) & set(va)) == 0 for tr, va in TEMPORAL_CV)
assert max(df.loc[SEARCH_FOLDS[-1]["valid"], "start_time"]) < CONFIRMATION_FOLD["lo"]      # the search folds end before the confirmation quarter starts
print("Search folds:", [f["fold"] for f in SEARCH_FOLDS], "| confirmation fold:", CONFIRMATION_FOLD["fold"], "| test period not loaded")

In [ ]:
log_decision("3", "Models to optimise", "Gradient boosting and random forest.",
             "Tied at the top of Task 3 (0.626 and 0.622, a difference of 0.003, below the tolerance of 0.005), with different ensemble principles; the other models are far from the best or insensitive to their one setting.")
log_decision("4", "Search design",
             "RandomizedSearchCV with an explicit space, scoring = PR-AUC, temporal splits of folds 1 to 3 (selection) and fold 4 kept for confirmation; 40 configurations for boosting and 20 for the forest; "
             "the cheapest configuration within the tolerance of the best is kept.",
             "A search over many configurations always looks better on the data it saw, so the comparison with the initial configuration is made on a fold that the search never used.")

---
## 5. The searches

Each search fits every sampled configuration on the three search folds and averages the PR-AUC of their validation quarters. `refit=False` because the final models are refitted
later, on the right data and with the selected configuration. The cells below are the slow part of the notebook (several minutes each).

The function `tidy_results` turns the `cv_results_` of a search into a readable table, and `select_configuration` applies the selection rule of Section 4.2.

In [ ]:
def tidy_results(search):
    """Readable table of a search: mean and standard deviation of the PR-AUC over the search folds, fit time and parameters."""
    r = pd.DataFrame(search.cv_results_)
    out = pd.DataFrame({"candidate": np.arange(len(r)), "PR-AUC mean": r["mean_test_score"], "std across folds": r["std_test_score"],
                        **{f"fold {f['fold']}": r[f"split{i}_test_score"] for i, f in enumerate(SEARCH_FOLDS)},
                        "fit seconds": r["mean_fit_time"]})
    for column in r.columns:
        if column.startswith("param_"):
            out[column.replace("param_clf__", "")] = r[column]
    return out.sort_values("PR-AUC mean", ascending=False).reset_index(drop=True)


def select_configuration(results, tolerance=TOLERANCE):
    """Highest mean PR-AUC; among the configurations within `tolerance` of it, the one with the shortest fit time."""
    best = results["PR-AUC mean"].max()
    tied = results[results["PR-AUC mean"] >= best - tolerance]
    return tied["fit seconds"].idxmin(), len(tied)


def clean_params(row, name):
    """Parameters of one sampled configuration as plain Python values, read from the original `params` of the search
    (a DataFrame would turn integers into decimals because of the None values)."""
    sampled = SEARCH[name].cv_results_["params"][int(row["candidate"])]
    return {key.replace("clf__", ""): (value.item() if hasattr(value, "item") else value) for key, value in sampled.items()}


SEARCH, RESULTS, OPTIMISED = {}, {}, {}


def run_search(name):
    started = time.time()
    search = RandomizedSearchCV(BUILDERS[name](), param_distributions=SPACES[name], n_iter=N_ITER[name], scoring="average_precision",
                                cv=TEMPORAL_CV, refit=False, random_state=SEED, n_jobs=1, return_train_score=False)
    search.fit(df[SELECTED], df[TARGET])
    SEARCH[name], RESULTS[name] = search, tidy_results(search)
    chosen, n_tied = select_configuration(RESULTS[name])
    OPTIMISED[name] = clean_params(RESULTS[name].loc[chosen], name)
    initial_mean = initial_scores.loc[[f["fold"] for f in SEARCH_FOLDS], f"{name} (initial)"].mean()
    print(f"{name}: {N_ITER[name]} configurations x {len(SEARCH_FOLDS)} folds fitted in {(time.time() - started) / 60:.1f} min")
    print(f"  best mean PR-AUC on the search folds: {RESULTS[name]['PR-AUC mean'].max():.4f} | initial configuration on the same folds: {initial_mean:.4f}")
    print(f"  configurations within the tolerance of the best: {n_tied} | selected (cheapest of them): row {chosen}, {RESULTS[name].loc[chosen, 'PR-AUC mean']:.4f}, "
          f"{RESULTS[name].loc[chosen, 'fit seconds']:.1f} s per fit")
    top = RESULTS[name].head(10).copy()
    top["selected"] = ["yes" if i == chosen else "" for i in top.index]
    display(top.style.hide(axis="index").format({c: "{:.4f}" for c in ["PR-AUC mean", "std across folds", "fold 1", "fold 2", "fold 3"]} | {"fit seconds": "{:.1f}"}))
    print("Selected configuration:", OPTIMISED[name])
    return chosen


chosen_boosting = run_search("Gradient boosting")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

In [ ]:
chosen_forest = run_search("Random forest")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 5.1 What did the search tell us?

Two figures help to judge the search itself. The first compares the scores of all sampled configurations with the score of the initial configuration on the same folds, and
shows the cost of each. The second, in the spirit of the lecture's *prioritise the key parameters*, measures which parameters actually move the score, with the Spearman correlation
between a parameter and the score of the sampled configurations.

In [ ]:
def numeric_view(name):
    """Numeric version of the sampled parameters, to correlate them with the score."""
    r = RESULTS[name]
    out = pd.DataFrame(index=r.index)
    for key in SPACES[name]:
        column = key.replace("clf__", "")
        values = r[column]
        if column == "max_depth":
            out[column] = [99 if v is None or (isinstance(v, float) and np.isnan(v)) else float(v) for v in values]       # none = unlimited
        elif column == "max_features" and name == "Random forest":
            out[column] = [{"sqrt": np.sqrt(len(SELECTED)) / len(SELECTED), "log2": np.log2(len(SELECTED)) / len(SELECTED)}.get(v, v) for v in values]
            out[column] = out[column].astype(float)
        else:
            out[column] = values.astype(float)
    return out


fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
palette = {"Gradient boosting": C_ABN, "Random forest": C_GREEN}
for name, color in palette.items():
    axes[0].scatter(RESULTS[name]["fit seconds"], RESULTS[name]["PR-AUC mean"], s=28, alpha=0.7, color=color, label=name)
    initial_mean = initial_scores.loc[[f["fold"] for f in SEARCH_FOLDS], f"{name} (initial)"].mean()
    axes[0].scatter([FIT_SECONDS[f"{name} (initial)"]], [initial_mean], marker="*", s=230, color=color, edgecolor="k", zorder=5)
axes[0].set_xscale("log"); axes[0].set_xlabel("seconds per fit (log scale)"); axes[0].set_ylabel("mean PR-AUC on the search folds")
axes[0].set_title("(a) Sampled configurations (stars: initial)"); axes[0].legend(fontsize=8, loc="lower right")

correlations = {}
for name in palette:
    numeric = numeric_view(name)
    correlations[name] = pd.Series({c: spearmanr(numeric[c], RESULTS[name]["PR-AUC mean"]).statistic for c in numeric.columns})
for ax, name in zip(axes[1:], palette):
    s = correlations[name].sort_values()
    ax.barh(s.index, s.values, color=[palette[name] if v > 0 else C_GREY for v in s.values])
    ax.axvline(0, color="k", lw=0.8); ax.set_xlim(-1, 1); ax.set_xlabel("Spearman correlation with the PR-AUC")
    ax.set_title(f"({'bc'[list(palette).index(name)]}) {name}: which parameters matter")
fig.tight_layout()
savefig(fig, "05_1_search_overview")

spread = pd.DataFrame({name: {"best": RESULTS[name]["PR-AUC mean"].max(), "median": RESULTS[name]["PR-AUC mean"].median(), "worst": RESULTS[name]["PR-AUC mean"].min(),
                              "initial configuration": initial_scores.loc[[f["fold"] for f in SEARCH_FOLDS], f"{name} (initial)"].mean(),
                              "configurations within the tolerance of the best": int((RESULTS[name]["PR-AUC mean"] >= RESULTS[name]["PR-AUC mean"].max() - TOLERANCE).sum())}
                       for name in palette}).T
display(spread.style.format({c: "{:.4f}" for c in ["best", "median", "worst", "initial configuration"]} | {"configurations within the tolerance of the best": "{:.0f}"}))
display(pd.DataFrame(correlations).round(2))

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

---
## 6. Initial against optimised configuration

The brief asks whether the optimisation produces a **meaningful and stable** improvement. We call an improvement *meaningful* when, on the confirmation fold (which the search never
saw), the PR-AUC of the optimised configuration exceeds the initial one by more than the tolerance of 0.005 and a paired bootstrap of the sessions of that fold gives an interval
that excludes zero. We call it *stable* when the optimised configuration is better in every fold. The comparison on the search folds is shown too, but it is
optimistic by construction (the search chose the configuration on those folds), so it cannot decide.

The optimised configurations are fitted again on the four folds, with the same protocol as the initial ones.

In [ ]:
t0 = time.time()
optimised_scores = pd.DataFrame({f"{name} (optimised)": run_folds(f"{name} (optimised)", lambda name=name: BUILDERS[name](**OPTIMISED[name])) for name in INITIAL})
print(f"optimised configurations fitted on the four folds in {time.time() - t0:.0f} s")

rows = []
for name in INITIAL:
    for f in FOLDS:
        ini, opt = initial_scores.loc[f["fold"], f"{name} (initial)"], optimised_scores.loc[f["fold"], f"{name} (optimised)"]
        rows.append({"model": name, "fold": f["fold"], "role": "confirmation" if f["fold"] == CONFIRMATION_FOLD["fold"] else "search", "initial": ini, "optimised": opt, "difference": opt - ini})
by_fold = pd.DataFrame(rows)
display(by_fold.pivot(index="model", columns="fold", values="difference").rename(columns=lambda c: f"fold {c}").style.format("{:+.4f}").set_caption("PR-AUC: optimised minus initial"))
display(by_fold.set_index(["model", "fold"]).style.format({"initial": "{:.4f}", "optimised": "{:.4f}", "difference": "{:+.4f}"}))

A paired bootstrap gives the uncertainty of the difference on the confirmation fold. Both configurations are scored on the same resampled sessions, so the noise that they share
(the quarter's difficulty) cancels. The same table gives the other metrics and the fit time on that fold.

In [ ]:
rng = np.random.default_rng(SEED)
y4 = df.loc[CONFIRMATION_FOLD["valid"], TARGET].to_numpy()
boot_index = [rng.integers(0, len(y4), len(y4)) for _ in range(N_BOOTSTRAP)]
verdict_rows, metric_rows = [], []
for name in INITIAL:
    p_ini, p_opt = OOF[f"{name} (initial)"][CONFIRMATION_FOLD["fold"]], OOF[f"{name} (optimised)"][CONFIRMATION_FOLD["fold"]]
    deltas = np.array([average_precision_score(y4[i], p_opt[i]) - average_precision_score(y4[i], p_ini[i]) for i in boot_index])
    low, high = np.percentile(deltas, [2.5, 97.5])
    delta4 = by_fold[(by_fold["model"] == name) & (by_fold["role"] == "confirmation")]["difference"].iloc[0]
    search_mean = by_fold[(by_fold["model"] == name) & (by_fold["role"] == "search")]["difference"].mean()
    all_positive = bool((by_fold[by_fold["model"] == name]["difference"] > 0).all())
    meaningful = bool(delta4 > TOLERANCE and low > 0)
    verdict_rows.append({"model": name, "difference on the search folds (mean)": search_mean, "difference on the confirmation fold": delta4,
                         "95% interval (paired bootstrap)": f"{low:+.4f} to {high:+.4f}", "above the tolerance": delta4 > TOLERANCE,
                         "interval excludes zero": low > 0, "better in all four folds": all_positive, "meaningful": meaningful})
    for stage, p in (("initial", p_ini), ("optimised", p_opt)):
        metric_rows.append({"model": name, "configuration": stage, **ranking_metrics(y4, p), "fit seconds": FIT_SECONDS[f"{name} ({stage})"]})
verdict = pd.DataFrame(verdict_rows).set_index("model")
display(verdict.style.format({"difference on the search folds (mean)": "{:+.4f}", "difference on the confirmation fold": "{:+.4f}"}))
display(pd.DataFrame(metric_rows).set_index(["model", "configuration"]).style.format({"PR-AUC": "{:.4f}", "lift": "{:.2f}", "ROC-AUC": "{:.4f}", "Brier": "{:.4f}", "log-loss": "{:.4f}", "fit seconds": "{:.1f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
x = np.arange(len(FOLDS))
for ax, name in zip(axes, INITIAL):
    ax.bar(x - 0.2, initial_scores[f"{name} (initial)"].to_numpy(), width=0.4, color=C_GREY, label="initial")
    ax.bar(x + 0.2, optimised_scores[f"{name} (optimised)"].to_numpy(), width=0.4, color=palette[name], label="optimised")
    ax.axvspan(2.5, 3.5, color=C_PURPLE, alpha=0.12)
    ax.text(3, 0.545, "confirmation", ha="center", fontsize=8, color=C_PURPLE)
    ax.set_xticks(x); ax.set_xticklabels([f"fold {f['fold']}" for f in FOLDS]); ax.set_ylim(0.5, 0.7); ax.set_title(f"{name}"); ax.legend(fontsize=8, loc="upper left")
axes[0].set_ylabel("PR-AUC")
fig.suptitle("Initial and optimised configuration, fold by fold (folds 1 to 3 selected the configuration)", fontsize=10, y=1.02)
fig.tight_layout()
savefig(fig, "06_1_initial_vs_optimised")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 6.1 Which configuration and which model go forward?

The rule is the one of Section 4: a configuration found by the search is adopted only if its improvement is meaningful; otherwise we keep the **initial** configuration, which is already validated and has
the same cost or lower. Among the two models, we keep the one with the higher mean PR-AUC over the four folds (the ranking rule of Task 2). If they are within the tolerance, we keep the cheaper one. The "strongest
probabilistic model" of the brief is the model that results from this choice: both models output probabilities, and we judge them by PR-AUC and by the probability metrics (Brier score, log-loss).

In [ ]:
FINAL_CONFIG = {name: (OPTIMISED[name] if verdict.loc[name, "meaningful"] else INITIAL[name]) for name in INITIAL}
FINAL_STAGE = {name: ("optimised" if verdict.loc[name, "meaningful"] else "initial") for name in INITIAL}
rows = []
for name in INITIAL:
    label = f"{name} ({FINAL_STAGE[name]})"
    s = summarise_folds(pr_auc_by_fold(label).to_numpy(), fold_table["validation sessions"].to_numpy())
    pooled_y = np.concatenate([df.loc[f["valid"], TARGET].to_numpy() for f in FOLDS])
    pooled_p = np.concatenate([OOF[label][f["fold"]] for f in FOLDS])
    m = ranking_metrics(pooled_y, pooled_p)
    rows.append({"model": name, "configuration kept": FINAL_STAGE[name], "PR-AUC mean": s["mean (unweighted)"], "std across folds": s["std across folds"], "worst fold": s["worst fold"],
                 "Brier (pooled)": m["Brier"], "log-loss (pooled)": m["log-loss"], "fit seconds": FIT_SECONDS[label]})
candidates = pd.DataFrame(rows).set_index("model").sort_values("PR-AUC mean", ascending=False)
display(candidates.style.format({"PR-AUC mean": "{:.4f}", "std across folds": "{:.4f}", "worst fold": "{:.4f}", "Brier (pooled)": "{:.4f}", "log-loss (pooled)": "{:.4f}", "fit seconds": "{:.1f}"}))
top_two = candidates["PR-AUC mean"].iloc[:2]
FINAL_NAME = candidates.index[0] if top_two.iloc[0] - top_two.iloc[1] > TOLERANCE else candidates.sort_values("fit seconds").index[0]
FINAL_LABEL = f"{FINAL_NAME} ({FINAL_STAGE[FINAL_NAME]})"
print(f"Final model: {FINAL_NAME} with its {FINAL_STAGE[FINAL_NAME]} configuration: {FINAL_CONFIG[FINAL_NAME]}")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

In [ ]:
log_decision("5", "Optimised configurations",
             "; ".join(f"{name}: {OPTIMISED[name]}" for name in INITIAL),
             "Selected on folds 1 to 3 with RandomizedSearchCV (PR-AUC, temporal splits); the cheapest configuration within the tolerance of the best.")
log_decision("6", "Is the optimisation worth it?",
             "; ".join(f"{name}: {'adopt the optimised configuration' if verdict.loc[name, 'meaningful'] else 'keep the initial configuration'}" for name in INITIAL),
             "Judged on fold 4, which the search never used: difference above the tolerance of 0.005, paired-bootstrap interval excluding zero, and sign in every fold.")
log_decision("6", "Final model", f"{FINAL_NAME}, {FINAL_STAGE[FINAL_NAME]} configuration.",
             "Highest mean PR-AUC over the four folds; a tie within the tolerance is resolved in favour of the cheaper model.")

---
## 7. Decision threshold and calibration

A model that outputs a probability becomes an alert system only when we choose a **threshold**: sessions with a probability at or above it raise an alert. The default of 0.50
treats both classes alike, which is not the case here. The positive class is only about 18.5% of the sessions, so few sessions reach a probability of 0.50, and Task 2 (Section 5)
argued that a missed failure costs more than a false alert as long as false alerts stay within what an operator can handle.

Following the brief, the threshold is chosen from **out-of-fold predictions**: for every development session, the probability comes from a model that was **not fitted on that session**
(each fold's model is fitted on the quarters before it). This is the only kind of prediction on which a threshold can be judged fairly, because the predictions of a model on its own training
data are too good. The test period plays no part in this section.

### 7.1 Out-of-fold predictions of the final model

In [ ]:
y_oof = np.concatenate([df.loc[f["valid"], TARGET].to_numpy() for f in FOLDS])
p_oof = np.concatenate([OOF[FINAL_LABEL][f["fold"]] for f in FOLDS])
fold_oof = np.concatenate([np.full(len(f["valid"]), f["fold"]) for f in FOLDS])
month_oof = np.concatenate([df.loc[f["valid"], "start_time"].dt.to_period("M").astype(str).to_numpy() for f in FOLDS])
print(f"{len(y_oof):,} out-of-fold predictions of '{FINAL_LABEL}' | abnormal share {y_oof.mean():.2%} | "
      f"mean predicted probability {p_oof.mean():.3f} | share of sessions with probability >= 0.50: {(p_oof >= 0.5).mean():.1%}")
assert len(y_oof) == fold_table["validation sessions"].sum()

### 7.2 What the default threshold of 0.50 does

The table gives the measures of the brief at the default threshold for all out-of-fold predictions and for each fold.

In [ ]:
rows = [{"data": "all four folds", **threshold_report(y_oof, p_oof, 0.5)}]
rows += [{"data": f"fold {f['fold']}", **threshold_report(y_oof[fold_oof == f["fold"]], p_oof[fold_oof == f["fold"]], 0.5)} for f in FOLDS]
default_table = pd.DataFrame(rows).set_index("data").drop(columns="threshold")
display(default_table.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}", "alert rate": "{:.1%}", "correct alerts per 1,000": "{:.1f}", "false alerts per 1,000": "{:.1f}", "missed per 1,000": "{:.1f}"}))

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 7.3 Choosing one criterion

The brief lists five criteria (maximise F1, reach a minimum recall, limit the false alerts, minimise a cost, respect an alert rate) and asks us to investigate **one**. We choose the one
that follows from the application defined in Task 2: alerts are reviewed by an operator with limited capacity, so what has to be limited is the **waste**, and within that limit we want to
catch as many failures as possible.

**Criterion: limit the false alerts.** We take the **lowest threshold at which the precision of the alerts is at least 50%**, that is, at most as many false alerts as correct alerts: an
operator wastes no more than one inspection for each real failure found. A lower threshold gives more recall, so the lowest threshold that keeps the precision at 50% is the one that catches the most
failures within the limit. The threshold is searched on a grid with a step of 0.01 on the pooled out-of-fold predictions.

The other criteria need an assumption that we cannot support with data: a minimum recall or an alert rate needs the capacity or the target of the operator, and a cost function needs the ratio between
the cost of a missed failure and of a false alert, which is unknown (Task 2, Section 5). The precision floor is also an assumption, but it is the easiest one to read and to discuss with an operator, and the
sensitivity to it is shown below. To make the choice visible, the table lists what two other criteria would give on the same predictions. They are not used afterwards.

In [ ]:
GRID = np.round(np.arange(0.05, 0.96, 0.01), 2)
curve = pd.DataFrame([threshold_report(y_oof, p_oof, t) for t in GRID]).set_index("threshold")

PRECISION_FLOOR = 0.50
t_floor = float(curve.index[curve["precision"] >= PRECISION_FLOOR].min())             # lowest threshold that keeps the precision at the floor
t_f1 = float(curve["F1"].idxmax())                                                   # reference criterion 1: maximise F1
t_budget = float((curve["alert rate"] - 0.10).abs().idxmin())                         # reference criterion 2: alert rate of about 10%
options = pd.DataFrame([{"criterion": "default threshold", **threshold_report(y_oof, p_oof, 0.5)},
                        {"criterion": "limit the false alerts: precision >= 50% (chosen)", **threshold_report(y_oof, p_oof, t_floor)},
                        {"criterion": "reference: maximise F1", **threshold_report(y_oof, p_oof, t_f1)},
                        {"criterion": "reference: alert rate of about 10%", **threshold_report(y_oof, p_oof, t_budget)}]).set_index("criterion")
display(options.style.format({"threshold": "{:.2f}", "precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}", "alert rate": "{:.1%}",
                              "correct alerts per 1,000": "{:.1f}", "false alerts per 1,000": "{:.1f}", "missed per 1,000": "{:.1f}"}))

sensitivity = pd.DataFrame([{"precision floor": floor, **threshold_report(y_oof, p_oof, float(curve.index[curve["precision"] >= floor].min()))} for floor in (0.40, 0.50, 0.60, 0.70)]).set_index("precision floor")
print("Sensitivity of the threshold to the precision floor:")
display(sensitivity.style.format({"threshold": "{:.2f}", "precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}", "alert rate": "{:.1%}",
                                  "correct alerts per 1,000": "{:.1f}", "false alerts per 1,000": "{:.1f}", "missed per 1,000": "{:.1f}"}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.6))
for column, color in [("precision", C_NORMAL), ("recall", C_ABN), ("F1", C_GREEN), ("alert rate", C_GREY)]:
    axes[0].plot(curve.index, curve[column], color=color, label=column, ls="--" if column == "alert rate" else "-")
axes[0].axvline(0.5, color="k", ls=":", label="default 0.50")
axes[0].axvline(t_floor, color=C_ORANGE, lw=2, label=f"selected {t_floor:.2f}")
axes[0].axhline(PRECISION_FLOOR, color=C_NORMAL, lw=0.8, ls=":")
axes[0].set_xlabel("threshold"); axes[0].set_ylabel("value"); axes[0].set_title("(a) Measures against the threshold (pooled out-of-fold predictions)"); axes[0].legend(fontsize=8)
precision_curve, recall_curve, _ = precision_recall_curve(y_oof, p_oof)
axes[1].plot(recall_curve, precision_curve, color=C_NORMAL)
for t, label, color in [(0.5, "default 0.50", "k"), (t_floor, f"selected {t_floor:.2f}", C_ORANGE), (t_f1, f"max F1 {t_f1:.2f}", C_GREEN), (t_budget, f"alert rate 10% ({t_budget:.2f})", C_PURPLE)]:
    r = threshold_report(y_oof, p_oof, t)
    axes[1].scatter([r["recall"]], [r["precision"]], s=70, color=color, zorder=5, label=label, edgecolor="k")
axes[1].axhline(PRECISION_FLOOR, color=C_NORMAL, lw=0.8, ls=":"); axes[1].axhline(y_oof.mean(), color=C_GREY, lw=0.8, ls="--")
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].set_ylim(0, 1.02); axes[1].set_title("(b) Precision-recall curve and operating points"); axes[1].legend(fontsize=8, loc="upper right")
fig.tight_layout()
savefig(fig, "07_1_threshold_choice")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 7.4 Is the selected threshold stable across periods?

The threshold was chosen on all folds pooled. The table shows what it does in each fold and in each month of the out-of-fold predictions. If the abnormal share of a period is
lower, the same probability threshold gives a lower precision there, and the alert rate moves too. The test period will meet its own abnormal share, and this table tells us how much
to expect that.

In [ ]:
THRESHOLD = t_floor
rows = [{"period": "all four folds", **threshold_report(y_oof, p_oof, THRESHOLD)}]
rows += [{"period": f"fold {f['fold']}", **threshold_report(y_oof[fold_oof == f["fold"]], p_oof[fold_oof == f["fold"]], THRESHOLD)} for f in FOLDS]
stability = pd.DataFrame(rows).set_index("period")
stability.insert(0, "abnormal share", [y_oof.mean()] + [y_oof[fold_oof == f["fold"]].mean() for f in FOLDS])
display(stability.drop(columns="threshold").style.format({"abnormal share": "{:.1%}", "precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}", "alert rate": "{:.1%}",
                                                          "correct alerts per 1,000": "{:.1f}", "false alerts per 1,000": "{:.1f}", "missed per 1,000": "{:.1f}"}))
monthly = pd.DataFrame([{"month": m, "sessions": int((month_oof == m).sum()), "abnormal share": y_oof[month_oof == m].mean(), **threshold_report(y_oof[month_oof == m], p_oof[month_oof == m], THRESHOLD)}
                        for m in sorted(set(month_oof))]).set_index("month")[["sessions", "abnormal share", "precision", "recall", "alert rate"]]
print("By month (out-of-fold):")
display(monthly.style.format({"sessions": "{:,.0f}", "abnormal share": "{:.1%}", "precision": "{:.3f}", "recall": "{:.3f}", "alert rate": "{:.1%}"}))
print(f"Correlation between the monthly abnormal share and the monthly precision at the selected threshold: {np.corrcoef(monthly['abnormal share'], monthly['precision'])[0, 1]:.2f}")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 7.5 Is calibration necessary?

The brief asks us to consider calibration if the outputs are used as probabilities. In our use they are not shown to anybody: they are compared with a threshold that we chose **empirically** on out-of-fold
predictions (the precision of the alerts is measured on real labels), so the threshold does not rely on the probabilities being exact. Calibration would matter in two cases: if a probability were reported
to an operator, and if the threshold came from a cost formula such as `t* = 1/(1 + c)` (Task 2, Section 5), which is only valid for calibrated probabilities. We still check it, because it tells us
whether the default threshold of 0.50 can be read as "a 50% chance of failure", and because the check is cheap.

Three views are used: a reliability table (predicted against observed frequency in ten groups of equal size), the calibration slope and intercept (a logistic regression of the label on the
logit of the prediction: slope 1 and intercept 0 are perfect), and a test of whether an isotonic recalibration would help, fitted on the out-of-fold predictions of folds 1 to 3 and applied to fold 4.

In [ ]:
bins = pd.qcut(p_oof, 10, duplicates="drop")
reliability = pd.DataFrame({"mean predicted": pd.Series(p_oof).groupby(bins, observed=True).mean(), "observed share": pd.Series(y_oof).groupby(bins, observed=True).mean(),
                            "sessions": pd.Series(y_oof).groupby(bins, observed=True).size()}).reset_index(drop=True)
ece = float(np.average(np.abs(reliability["mean predicted"] - reliability["observed share"]), weights=reliability["sessions"]))
logit = np.log(np.clip(p_oof, 1e-4, 1 - 1e-4) / (1 - np.clip(p_oof, 1e-4, 1 - 1e-4)))
slope_model = LogisticRegression(C=1e6, max_iter=1000).fit(logit.reshape(-1, 1), y_oof)
slope, intercept = float(slope_model.coef_[0, 0]), float(slope_model.intercept_[0])
fit_idx, test_idx = fold_oof <= 3, fold_oof == 4
iso = IsotonicRegression(out_of_bounds="clip").fit(p_oof[fit_idx], y_oof[fit_idx])
p4, p4_iso = p_oof[test_idx], iso.predict(p_oof[test_idx])
calibration = pd.DataFrame({"raw": [brier_score_loss(y_oof[test_idx], p4), log_loss(y_oof[test_idx], np.clip(p4, 1e-6, 1 - 1e-6)), average_precision_score(y_oof[test_idx], p4)],
                            "isotonic (fitted on folds 1 to 3)": [brier_score_loss(y_oof[test_idx], p4_iso), log_loss(y_oof[test_idx], np.clip(p4_iso, 1e-6, 1 - 1e-6)),
                                                                 average_precision_score(y_oof[test_idx], p4_iso)]}, index=["Brier (fold 4)", "log-loss (fold 4)", "PR-AUC (fold 4)"])
print(f"Expected calibration error (10 groups): {ece:.4f} | calibration slope {slope:.3f} | intercept {intercept:+.3f}")
display(reliability.style.format({"mean predicted": "{:.3f}", "observed share": "{:.3f}", "sessions": "{:,.0f}"}))
display(calibration.style.format("{:.4f}"))

fig, ax = plt.subplots(figsize=(5.4, 5))
ax.plot([0, 1], [0, 1], color=C_GREY, ls="--", label="perfect calibration")
ax.plot(reliability["mean predicted"], reliability["observed share"], marker="o", color=C_ABN, label=FINAL_NAME)
ax.set_xlabel("mean predicted probability"); ax.set_ylabel("observed share of abnormal sessions"); ax.set_title("Reliability of the out-of-fold probabilities"); ax.legend(fontsize=8)
savefig(fig, "07_2_calibration")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 7.6 Freezing everything before the test period

Everything that Section 8 will use is now fixed: the model, its hyperparameters, the features, the criterion and the threshold. We save it in a file and compute its SHA-256 checksum. After the test evaluation
the checksum is computed again, and the notebook stops if the file has changed. This is the practical meaning of "the threshold must not be selected or modified after inspecting the test results".

In [ ]:
FROZEN = {"model": FINAL_NAME, "configuration": FINAL_STAGE[FINAL_NAME],
          "hyperparameters": {k: (None if v is None else v) for k, v in FINAL_CONFIG[FINAL_NAME].items()},
          "features": SELECTED, "default_threshold": 0.5,
          "threshold_criterion": f"lowest threshold with precision >= {PRECISION_FLOOR:.0%} on the pooled out-of-fold predictions of the four development folds (grid step 0.01)",
          "threshold": THRESHOLD, "calibration": "none (the threshold is empirical; see Section 7.5)",
          "selected_on": "development data only (2020-04-01 to 2021-09-30)", "random_seed": SEED}
FROZEN_FILE = OUT_DIR / "task4_frozen.json"
FROZEN_FILE.write_text(json.dumps(FROZEN, indent=2, ensure_ascii=False, default=json_default))
FROZEN_HASH = hashlib.sha256(FROZEN_FILE.read_bytes()).hexdigest()
print(json.dumps(FROZEN, indent=2, ensure_ascii=False, default=json_default))
print("\nSHA-256 of the frozen file:", FROZEN_HASH)

In [ ]:
log_decision("7", "Threshold criterion",
             f"Limit the false alerts: the lowest threshold with precision >= {PRECISION_FLOOR:.0%} on the pooled out-of-fold predictions of the development folds, grid step 0.01. Selected threshold: {THRESHOLD:.2f}.",
             "The application is capacity-limited, so waste is capped and recall maximised within the cap; recall targets, alert rates and costs would need assumptions that the data cannot support (Task 2, Section 5).")
log_decision("7", "Calibration", "No recalibration.",
             f"Expected calibration error {ece:.4f}, slope {slope:.2f}; the threshold is empirical and the probabilities are not reported (Section 7.5).")
log_decision("7", "Frozen before the test", f"Model, configuration, features and threshold saved in {FROZEN_FILE.name} with a SHA-256 checksum; checked again after the test evaluation.",
             "The threshold must not be selected or modified after inspecting the final test results (Task 4 brief).")

---
## 8. Final test evaluation

Everything is frozen (Section 7.6). This section does three things, in this order: it refits the final model on **all** development data, loads the test sessions for the first and only time,
and applies the frozen model and threshold unchanged. Nothing computed from the test sessions is used to change a choice. The notebook checks the frozen file before and after.

### 8.1 Refitting the final model on all development data

As planned in Task 2 (Section 7.5), the final model is fitted on all labelled development sessions, those that **started and ended** before the test period (the sessions that
were still running on 2021-10-01 had no label at that moment and are removed, as in every fold). For the comparison of Section 8.2 the initial configuration of the same model is refitted in the same way.

In [ ]:
assert hashlib.sha256(FROZEN_FILE.read_bytes()).hexdigest() == FROZEN_HASH, "the frozen file changed before the test evaluation"
final_idx = df.index[df["end_time"] < TEST_START].to_numpy()
print(f"Training sessions: {len(final_idx):,} of {len(df):,} development sessions ({len(df) - len(final_idx)} purged because they ended after {TEST_START.date()})")

started = time.time()
final_model = BUILDERS[FINAL_NAME](**FINAL_CONFIG[FINAL_NAME]).fit(df.loc[final_idx, SELECTED], df.loc[final_idx, TARGET])
final_seconds = time.time() - started
initial_model = BUILDERS[FINAL_NAME](**INITIAL[FINAL_NAME]).fit(df.loc[final_idx, SELECTED], df.loc[final_idx, TARGET])
dev_prior = df.loc[final_idx, TARGET].mean()
print(f"Final model ({FINAL_NAME}, {FINAL_STAGE[FINAL_NAME]} configuration) fitted in {final_seconds:.1f} s")

### 8.2 Loading the test period, once

The sessions that start from 2021-10-01 are read now. They were not in memory in any earlier cell. The history attributes of the test sessions are the supplied ones, computed only from sessions
that had finished before each start (Task 1, Section 5), so using them is the same as using them in the development folds.

In [ ]:
test = load_sessions("test")
y_test = test[TARGET].to_numpy()
assert test["start_time"].min() >= TEST_START and len(test) > 0
print(f"Test period: {len(test):,} sessions ({test['start_time'].min().date()} to {test['start_time'].max().date()}), abnormal share {y_test.mean():.2%} "
      f"(development: {df[TARGET].mean():.2%})")

p_final = final_model.predict_proba(test[SELECTED])[:, 1]
p_initial = initial_model.predict_proba(test[SELECTED])[:, 1]
p_b1 = test["post_previous_abnormal_rate_7D"].to_numpy()
rng_test = np.random.default_rng(SEED)
p_b0_rank = rng_test.random(len(test))

### 8.3 Ranking quality on the test period

The first table repeats, on the test period, the comparison of Tasks 2 and 3 (the baselines B0 and B1) and of Section 6 (initial against final configuration). It uses the metrics that do not
depend on a threshold, and the alert-budget measures of Task 2 for the two models that matter.

In [ ]:
rows = []
for label, score, prob in [("B0 no skill", p_b0_rank, np.full(len(test), dev_prior)), ("B1 post failure rate (7 days)", p_b1, p_b1),
                           (f"{FINAL_NAME} (initial configuration)", p_initial, p_initial), (f"{FINAL_NAME} (final: {FINAL_STAGE[FINAL_NAME]})", p_final, p_final)]:
    m = {"PR-AUC": average_precision_score(y_test, score), "ROC-AUC": roc_auc_score(y_test, score),
         "Brier": brier_score_loss(y_test, prob), "log-loss": log_loss(y_test, np.clip(prob, 1e-6, 1 - 1e-6))}
    m["lift"] = m["PR-AUC"] / y_test.mean()
    rows.append({"model": label, **m})
test_ranking = pd.DataFrame(rows).set_index("model")[["PR-AUC", "lift", "ROC-AUC", "Brier", "log-loss"]]
display(test_ranking.style.format({"PR-AUC": "{:.4f}", "lift": "{:.2f}", "ROC-AUC": "{:.4f}", "Brier": "{:.4f}", "log-loss": "{:.4f}"}))

rows = []
for label, score in [("B1 post failure rate (7 days)", p_b1), (f"{FINAL_NAME} (final)", p_final)]:
    for q in ALERT_BUDGETS:
        rows.append({"model": label, "alert budget": f"{q:.0%}", **alert_metrics(y_test, score, q, np.random.default_rng(SEED))})
test_budget = pd.DataFrame(rows).set_index(["model", "alert budget"])
display(test_budget.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "correct alerts per 1,000": "{:.1f}", "false alerts per 1,000": "{:.1f}", "missed per 1,000": "{:.1f}"}))

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 8.4 The threshold on the test period: default against selected

This is the comparison that the brief asks for. The same probabilities are cut at the default threshold of 0.50 and at the frozen threshold, and the seven measures of the brief are compared. The out-of-fold
development values are repeated next to the test values: the difference between the two rows of a threshold shows how much the behaviour changes when the model meets new data.

In [ ]:
rows = []
for data, y, p in [("development, out-of-fold", y_oof, p_oof), ("test period", y_test, p_final)]:
    for name, thr in [("default", 0.5), ("selected", THRESHOLD)]:
        rows.append({"data": data, "threshold": f"{name} ({thr:.2f})", **{k: v for k, v in threshold_report(y, p, thr).items() if k != "threshold"}})
test_threshold = pd.DataFrame(rows).set_index(["data", "threshold"])
fmt = {"precision": "{:.3f}", "recall": "{:.3f}", "F1": "{:.3f}", "alert rate": "{:.1%}", "correct alerts per 1,000": "{:.1f}", "false alerts per 1,000": "{:.1f}", "missed per 1,000": "{:.1f}"}
display(test_threshold.style.format(fmt))

change = pd.DataFrame({data: test_threshold.loc[(data, f"selected ({THRESHOLD:.2f})")] - test_threshold.loc[(data, "default (0.50)")] for data in ("development, out-of-fold", "test period")}).T
print("Change from the default to the selected threshold (selected minus default):")
display(change.style.format({"precision": "{:+.3f}", "recall": "{:+.3f}", "F1": "{:+.3f}", "alert rate": "{:+.1%}", "correct alerts per 1,000": "{:+.1f}", "false alerts per 1,000": "{:+.1f}", "missed per 1,000": "{:+.1f}"}))

# the same results as a daily workload for the operator
days = (test["start_time"].max().normalize() - test["start_time"].min().normalize()).days + 1
daily = pd.DataFrame({name: {"sessions per day": len(test) / days,
                             "alerts per day": test_threshold.loc[("test period", name), "alert rate"] * len(test) / days,
                             "correct alerts per day": test_threshold.loc[("test period", name), "correct alerts per 1,000"] * len(test) / days / 1000,
                             "false alerts per day": test_threshold.loc[("test period", name), "false alerts per 1,000"] * len(test) / days / 1000,
                             "missed abnormal sessions per day": test_threshold.loc[("test period", name), "missed per 1,000"] * len(test) / days / 1000}
                      for name in (f"default (0.50)", f"selected ({THRESHOLD:.2f})")})
print(f"Daily workload on the test period ({days} days, all posts together):")
display(daily.style.format("{:.0f}"))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))
measures = ["precision", "recall", "F1"]
x = np.arange(len(measures))
for i, (name, thr, color) in enumerate([("default", 0.5, C_GREY), ("selected", THRESHOLD, C_ORANGE)]):
    r = test_threshold.loc[("test period", f"{name} ({thr:.2f})")]
    axes[0].bar(x + (i - 0.5) * 0.38, [r[m] for m in measures], width=0.38, color=color, label=f"{name} ({thr:.2f})")
    for xi, m in zip(x, measures):
        axes[0].text(xi + (i - 0.5) * 0.38, r[m] + 0.01, f"{r[m]:.2f}", ha="center", fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels(measures); axes[0].set_ylim(0, 1); axes[0].set_title("(a) Test period: precision, recall and F1"); axes[0].legend(fontsize=8)
counts = ["correct alerts per 1,000", "false alerts per 1,000", "missed per 1,000"]
x = np.arange(len(counts))
for i, (name, thr, color) in enumerate([("default", 0.5, C_GREY), ("selected", THRESHOLD, C_ORANGE)]):
    r = test_threshold.loc[("test period", f"{name} ({thr:.2f})")]
    axes[1].bar(x + (i - 0.5) * 0.38, [r[m] for m in counts], width=0.38, color=color, label=f"{name} ({thr:.2f})")
    for xi, m in zip(x, counts):
        axes[1].text(xi + (i - 0.5) * 0.38, r[m] + 2, f"{r[m]:.0f}", ha="center", fontsize=8)
axes[1].set_xticks(x); axes[1].set_xticklabels(["correct alerts", "false alerts", "missed abnormal"]); axes[1].set_ylabel("per 1,000 sessions"); axes[1].set_title("(b) Test period: alerts per 1,000 sessions"); axes[1].legend(fontsize=8)
fig.tight_layout()
savefig(fig, "08_1_threshold_on_test")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 8.5 Does the behaviour hold within the test period?

The test quarter has three months. The table shows the frozen threshold month by month. It is descriptive: nothing is adjusted from it. Its purpose is the operational question of whether the alert
rate and the precision stay put while the abnormal share moves.

In [ ]:
months = test["start_time"].dt.to_period("M").astype(str).to_numpy()
monthly_test = pd.DataFrame([{"month": m, "sessions": int((months == m).sum()), "abnormal share": y_test[months == m].mean(),
                              **{k: v for k, v in threshold_report(y_test[months == m], p_final[months == m], THRESHOLD).items() if k != "threshold"}}
                             for m in sorted(set(months))]).set_index("month")
display(monthly_test.style.format({"sessions": "{:,.0f}", "abnormal share": "{:.1%}", **fmt}))

### 8.6 Why is the test quarter harder than the development quarters?

The test scores are lower than the development scores (Section 8.3 against Section 6), and the precision at the selected threshold is below its target. Before reading that as a failure of the model, the
table compares the test quarter with the four development quarters using only things that involve no fitting and no choice: the baseline B1 (which learns nothing, so it cannot have overfitted the development folds),
the model's out-of-fold or test predictions, and the abnormal share inside three groups of sessions defined by the 7-day failure rate of their post at the start of the session. It is descriptive and changes nothing
that was decided.

In [ ]:
def profile(name, d, y, p_model):
    r7 = d["post_previous_abnormal_rate_7D"].to_numpy()
    band = np.select([r7 < 0.10, r7 < 0.30], ["low", "medium"], default="high")
    return {"period": name, "abnormal share": y.mean(), "B1 PR-AUC": average_precision_score(y, r7), "model PR-AUC": average_precision_score(y, p_model),
            "gain of the model over B1": average_precision_score(y, p_model) - average_precision_score(y, r7),
            "sessions with a high 7-day rate": (band == "high").mean(), "abnormal share, low band (<0.10)": y[band == "low"].mean(),
            "abnormal share, medium band": y[band == "medium"].mean(), "abnormal share, high band (>=0.30)": y[band == "high"].mean()}


rows = [profile(f"development, fold {f['fold']}", df.loc[f["valid"]], df.loc[f["valid"], TARGET].to_numpy(), OOF[FINAL_LABEL][f["fold"]]) for f in FOLDS]
rows.append(profile("test period", test, y_test, p_final))
period_profile = pd.DataFrame(rows).set_index("period")
display(period_profile.style.format({"abnormal share": "{:.1%}", "B1 PR-AUC": "{:.3f}", "model PR-AUC": "{:.3f}", "gain of the model over B1": "{:+.3f}", "sessions with a high 7-day rate": "{:.1%}",
                                     "abnormal share, low band (<0.10)": "{:.1%}", "abnormal share, medium band": "{:.1%}", "abnormal share, high band (>=0.30)": "{:.1%}"}))

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

In [ ]:
assert hashlib.sha256(FROZEN_FILE.read_bytes()).hexdigest() == FROZEN_HASH, "the frozen file changed after the test evaluation"
assert json.loads(FROZEN_FILE.read_text())["threshold"] == THRESHOLD
print("Integrity check passed: the frozen file (model, hyperparameters, features, threshold) is identical before and after the test evaluation.")
log_decision("8", "Test evaluation", "One evaluation of the refitted final model, with the frozen threshold applied unchanged; the initial configuration and the baselines are scored for reference only.",
             "SHA-256 checksum of the frozen file identical before loading the test and after the evaluation.")

---
## 9. Interpretation of the final model

### 9.1 Which method, and why

The final model is an ensemble of trees. It has no coefficients to read, and the importances that the trees compute while they are built (impurity importances) are biased towards variables
with many distinct values and are not available for `HistGradientBoostingClassifier`. We use two methods that work for any model and that measure what the **predictions** depend on:

* **Permutation importance, in groups.** The values of a variable are shuffled among the sessions of a held-out set and we measure how much the PR-AUC falls: if the model relies on the variable, the
  score drops. Our variables are strongly correlated inside a group (the 1-, 7- and 30-day windows describe the same events), and a model can recover a shuffled window from its neighbours, so one-variable
  importances understate each of them. We therefore shuffle **a whole feature group at once** (the same row permutation for all its columns, which keeps the relations inside the group) and report the groups
  of Task 3 first. Single variables are shown afterwards.
* **Partial dependence.** For a chosen variable it shows the average predicted probability as that variable is moved across its range, with the others as observed. It describes the *shape* of the effect (rising,
  falling, U-shaped), which importance does not.

**Which data.** The final model is fitted on all development data, so its predictions on development data are too good to interpret. We therefore interpret the **same configuration fitted on the
training window of fold 4** (everything before 2021-07-01) on its validation quarter, which it has not seen. It is the closest development model to the final one. We repeat the grouped importance
for the final model on the test period as a descriptive check; it changes nothing that was decided. Both use a fixed random subsample of 30,000 sessions to keep the computing time reasonable.

In [ ]:
GROUP_COLUMNS = {g: [c for c in cols if c in SELECTED] for g, cols in GROUPS.items()}
GROUP_COLUMNS = {g: cols for g, cols in GROUP_COLUMNS.items() if cols}          # the tariff group was dropped in Task 3
assert sorted(c for cols in GROUP_COLUMNS.values() for c in cols) == sorted(SELECTED)

model_dev = BUILDERS[FINAL_NAME](**FINAL_CONFIG[FINAL_NAME]).fit(df.loc[CONFIRMATION_FOLD["train"], SELECTED], df.loc[CONFIRMATION_FOLD["train"], TARGET])
sample_dev = np.random.default_rng(SEED).choice(CONFIRMATION_FOLD["valid"], 30_000, replace=False)
X_dev, y_dev = df.loc[sample_dev, SELECTED], df.loc[sample_dev, TARGET].to_numpy()
sample_test = np.random.default_rng(SEED).choice(len(test), 30_000, replace=False)
X_tst, y_tst = test.iloc[sample_test][SELECTED].reset_index(drop=True), y_test[sample_test]
assert np.allclose(model_dev.predict_proba(df.loc[CONFIRMATION_FOLD["valid"][:200], SELECTED])[:, 1], OOF[FINAL_LABEL][CONFIRMATION_FOLD["fold"]][:200])   # same model as the fold-4 predictions


def permutation_drop(model, X, y, columns, repeats, seed=SEED):
    """Mean and standard deviation of the fall in PR-AUC when the rows of `columns` are shuffled together (same permutation for all of them)."""
    rng = np.random.default_rng(seed)
    base = average_precision_score(y, model.predict_proba(X)[:, 1])
    drops = []
    for _ in range(repeats):
        perm = rng.permutation(len(X))
        shuffled = X.copy()
        for c in columns:
            shuffled[c] = X[c].to_numpy()[perm]
        drops.append(base - average_precision_score(y, model.predict_proba(shuffled)[:, 1]))
    return float(np.mean(drops)), float(np.std(drops, ddof=1)), base


t0 = time.time()
importance = {}
for name, model, X, y in [("development (fold 4 model)", model_dev, X_dev, y_dev), ("test (final model)", final_model, X_tst, y_tst)]:
    groups_drop = {g: permutation_drop(model, X, y, cols, repeats=5) for g, cols in GROUP_COLUMNS.items()}
    features_drop = {c: permutation_drop(model, X, y, [c], repeats=3) for c in SELECTED}
    importance[name] = {"groups": pd.DataFrame(groups_drop, index=["drop", "std", "baseline PR-AUC"]).T, "features": pd.DataFrame(features_drop, index=["drop", "std", "baseline PR-AUC"]).T}
print(f"permutation importance computed in {time.time() - t0:.0f} s")

group_table = pd.DataFrame({name: v["groups"]["drop"] for name, v in importance.items()}).sort_values("development (fold 4 model)", ascending=False)
group_table["share of the sum (development)"] = group_table["development (fold 4 model)"] / group_table["development (fold 4 model)"].sum()
print(f"Baseline PR-AUC of the 30,000-session samples: development {importance['development (fold 4 model)']['groups']['baseline PR-AUC'].iloc[0]:.4f}, "
      f"test {importance['test (final model)']['groups']['baseline PR-AUC'].iloc[0]:.4f}")
print("Fall in PR-AUC when a whole feature group is shuffled:")
display(group_table.style.format({"development (fold 4 model)": "{:.4f}", "test (final model)": "{:.4f}", "share of the sum (development)": "{:.1%}"}))
feature_table = importance["development (fold 4 model)"]["features"].sort_values("drop", ascending=False)
feature_table["group"] = [next(g for g, cols in GROUP_COLUMNS.items() if c in cols) for c in feature_table.index]
feature_table["fall on test"] = importance["test (final model)"]["features"]["drop"].reindex(feature_table.index)
print("Fall in PR-AUC when one variable is shuffled (ten largest, development):")
display(feature_table.head(10)[["group", "drop", "std", "fall on test"]].style.format({"drop": "{:.4f}", "std": "{:.4f}", "fall on test": "{:.4f}"}))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
y_pos = np.arange(len(group_table))[::-1]
axes[0].barh(y_pos + 0.2, group_table["development (fold 4 model)"], height=0.38, color=C_ABN, label="development (fold 4 model)",
             xerr=importance["development (fold 4 model)"]["groups"].loc[group_table.index, "std"])
axes[0].barh(y_pos - 0.2, group_table["test (final model)"], height=0.38, color=C_GREEN, label="test (final model)",
             xerr=importance["test (final model)"]["groups"].loc[group_table.index, "std"])
axes[0].set_yticks(y_pos); axes[0].set_yticklabels(group_table.index); axes[0].set_xlabel("fall in PR-AUC when the group is shuffled"); axes[0].set_title("(a) Feature groups"); axes[0].legend(fontsize=8)
top = feature_table.head(12)[::-1]
axes[1].barh(top.index, top["drop"], xerr=top["std"], color=C_NORMAL)
axes[1].set_xlabel("fall in PR-AUC when the variable is shuffled"); axes[1].set_title("(b) The twelve most important single variables (development)")
fig.tight_layout()
savefig(fig, "09_1_permutation_importance")

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 9.2 The shape of the effects: partial dependence

Partial dependence is computed for the four numeric variables with the largest single-variable importance, on a random sample of 20,000 sessions of the validation quarter of fold 4, with the fold 4 model.
The horizontal dashed line is the abnormal share of that quarter.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

numeric_top = [c for c in feature_table.index if c not in FEATURE_GROUPS["categorical"] and c not in ("hour", "dow")][:4]
pd_sample = df.loc[np.random.default_rng(SEED + 1).choice(CONFIRMATION_FOLD["valid"], 20_000, replace=False), SELECTED]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.9), sharey=True)
PartialDependenceDisplay.from_estimator(model_dev, pd_sample, features=numeric_top, kind="average", grid_resolution=25, percentiles=(0.02, 0.98), ax=axes, line_kw={"color": C_ABN})
for ax in axes:
    ax.axhline(y_dev.mean(), color=C_GREY, ls="--", lw=0.8)
axes[0].set_ylabel("average predicted probability")
fig.suptitle("Partial dependence of the final configuration (fold 4 model, validation quarter of fold 4)", fontsize=10, y=1.03)
fig.tight_layout()
savefig(fig, "09_2_partial_dependence")
print("Variables shown:", numeric_top)

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

### 9.3 What these methods cannot tell us

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

---
## 10. Synthesis, saved files and decision log

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*

In [ ]:
results = {
    "models_optimised": list(INITIAL),
    "search": {name: {"method": "RandomizedSearchCV", "scoring": "average_precision", "n_iter": N_ITER[name], "search_folds": [f["fold"] for f in SEARCH_FOLDS],
                      "confirmation_fold": CONFIRMATION_FOLD["fold"], "tolerance": TOLERANCE,
                      "space": space_table[space_table["model"] == name][["parameter", "distribution", "range"]].to_dict("records"),
                      "best_search_score": float(RESULTS[name]["PR-AUC mean"].max()),
                      "initial_search_score": float(initial_scores.loc[[f["fold"] for f in SEARCH_FOLDS], f"{name} (initial)"].mean()),
                      "selected_parameters": OPTIMISED[name], "initial_parameters": INITIAL[name]} for name in INITIAL},
    "initial_vs_optimised": {"by_fold": by_fold.to_dict("records"), "verdict": verdict.reset_index().to_dict("records")},
    "final_model": {"name": FINAL_NAME, "configuration": FINAL_STAGE[FINAL_NAME], "hyperparameters": FINAL_CONFIG[FINAL_NAME]},
    "threshold": {"criterion": FROZEN["threshold_criterion"], "default": 0.5, "selected": THRESHOLD, "calibration": {"ece": ece, "slope": slope, "intercept": intercept},
                  "development_out_of_fold": {k: float(v) for k, v in test_threshold.loc[("development, out-of-fold", f"selected ({THRESHOLD:.2f})")].items()},
                  "test": {k: float(v) for k, v in test_threshold.loc[("test period", f"selected ({THRESHOLD:.2f})")].items()}},
    "test_ranking": test_ranking.reset_index().to_dict("records"),
    "importance_groups": group_table.reset_index().rename(columns={"index": "group"}).to_dict("records"),
    "frozen_file": {"path": str(FROZEN_FILE), "sha256": FROZEN_HASH},
}
(OUT_DIR / "task4_results.json").write_text(json.dumps(results, indent=2, ensure_ascii=False, default=json_default))
for name in INITIAL:
    RESULTS[name].to_csv(OUT_DIR / f"task4_search_{name.lower().replace(' ', '_')}.csv", index=False)
test_threshold.reset_index().to_csv(OUT_DIR / "task4_threshold_comparison.csv", index=False)
decision_log = pd.DataFrame(DECISIONS)
decision_log.to_csv(OUT_DIR / "task4_decision_log.csv", index=False)
assert hashlib.sha256(FROZEN_FILE.read_bytes()).hexdigest() == FROZEN_HASH
print("Saved: task4_results.json, task4_frozen.json, task4_threshold_comparison.csv, the two search tables, task4_decision_log.csv | figures in", FIG_DIR)
display(decision_log.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

### Limitations and assumptions that remain open

*The commentary on this result is written after the full run of the notebook, once the searches have finished and the numbers exist.*